# Topic Analysis with Fine-Tuned BERT
In this notebook, we perform and describe the steps for fine-tuning a pre-trained BERT uncased model for the topic analysis task.

## Install/import libraries and prepare Colab environment

In [31]:
import pandas as pd
import numpy as np
import sklearn
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split

In [7]:
# Clone the repository into the Colab environment
!git clone https://github.com/einavrobyncohen/Text-Mining-Assignments.git

# Set the working directory
%cd Text-Mining-Assignments/tmproject

/content/Text-Mining-Assignments/tmproject


## Data exploration

We load the test and training datasets. Since we perform only topic analysis in this notebook, we drop the column titled "sentiment" from each training dataset.

In [32]:
# Load training dataset, dropping the sentiment column
path_to_train_folder = "training_sets/sentiment-topic/"
path_to_train_file = f"{path_to_train_folder}sentiment-topic-train.tsv"
train = pd.read_csv(
    path_to_train_file,
    sep="\t",
    usecols=["sentence", "topic"]
)

# Load test dataset, dropping the sentiment column
path_to_test_folder = "test_sets/"
path_to_test_file = f"{path_to_test_folder}sentiment-topic-test.tsv"
test = pd.read_csv(
    path_to_test_file,
    sep="\t",
    usecols=["sentence", "topic"]
)

We inspect whether the training data contains null values.

In [9]:
# Check the training data for null values
print(f"{train['topic'].isnull().value_counts()}\n")
print(train['sentence'].isnull().value_counts())

topic
False    35778
Name: count, dtype: int64

sentence
False    35778
Name: count, dtype: int64


We inspect the first few rows and value distribution of the training dataset to have a general idea of the kind of data we are dealing with.

In [10]:
# Print the number of instances per topic in the training set
print(train['topic'].value_counts())

# Inspect the first few rows of the training data
train.head(5)

topic
sports    17490
book      16587
movie      1701
Name: count, dtype: int64


,sentence,topic
0,Aggie is angela lansbury who carries pocketboo...,book
1,While this light murder mystery is laced with ...,book
2,"The setting, views described in the story are ...",book
3,Very light reading.,book
4,I did not expect this type of book to be in li...,book


The number of instances labelled with the topic "movie" is significantly lower than those labelled with another topic. This class imbalance will be accounted for.

In [11]:
# Print the number of instances per topic in the test set
print(test['topic'].value_counts())

# Inspect the first few rows of the test data
test.head(5)

topic
sports    6
book      6
movie     6
Name: count, dtype: int64


,sentence,topic
0,The atmosphere at the stadium tonight was elec...,sports
1,The game was so intense I forgot to breathe at...,sports
2,It had me hooked from the first chapter.,book
3,"It’s more of a slow burn than a page-turner, b...",book
4,"It’s split into two timelines, which keeps it ...",book


## Train test split

We split our training data into a training set and a validation set. Since the training dataset has class imbalance, we use stratified sampling. This ensures that the class distribution is maintained in the training and validation sets.

Each dataset contains the topics "book", "movie", and "sports". We assign these integers 0, 1, and 2, respectively. The target attribute is the integer index of the topic.

For efficiency purposes, we convert the train, dev, and test data to the HuggingFace dataset format.

In [33]:
# Validation selection, 10% of training data
train, dev = train_test_split(
    train,
    test_size=0.1,
    random_state=0,
    stratify=train[['topic']]
)

# Create mapping for topic labels
topic_mapping = {"book": 0, "movie": 1, "sports": 2}

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(train)
dev_dataset = Dataset.from_pandas(dev)
test_dataset = Dataset.from_pandas(test)

## Tokenization and preprocessing
The transformers library contains the BERT-base-uncased tokenizer. We use this tokenizer to prepare the text data for the BERT model by performing the following steps.
* Convert all text to lowercase ("uncased")
* Apply WordPiece tokenization to handle words that do not appear in the vocabulary
* Add special tokens, such as [CLS] and [SEP].
* Pad/truncate all sequences to a maximum length of 128 tokens. This maximum sequence is sufficient for most sentences and computationally efficient.

In [34]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def preprocess_function(examples):
    """
    Prepares the text data for the BERT model by tokenizing it.
    """
    return tokenizer(
        examples['sentence'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

# Tokenize and prepare datasets
train_dataset = train_dataset.map(preprocess_function, batched=True)
dev_dataset = dev_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

# Set correct format
train_dataset = train_dataset.map(
    lambda examples: {'labels': topic_mapping[examples['topic']]}
)
dev_dataset = dev_dataset.map(
    lambda examples: {'labels': topic_mapping[examples['topic']]}
)
test_dataset = test_dataset.map(
    lambda examples: {'labels': topic_mapping[examples['topic']]}
)

# Set columns format
train_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)
dev_dataset.set_format(
    type='torch', columns=['input_ids', 'attention_mask', 'labels']
)
test_dataset.set_format(
    type='torch', columns=['input_ids', 'attention_mask', 'labels']
)


Map:   0%|          | 0/32200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3578 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Map:   0%|          | 0/32200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3578 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

## Define model's configuration

We load the pre-trained BERT uncased model and define the model's configuration using the hyperparameters found by Geetha and Renuka (2021).

Key configurations include:
* BERT-base-uncased with 110M parameters and a classification head for 3 topic classes
* Learning rate of 2e-5 and 5 training epochs based on Geetha and Renuka's (citation below) optimal parameters for sentiment analysis.
* Weighted F1-score as optimization metric to address class imbalance.
* Model evaluation and checkpointing after each epoch.

> M.P. Geetha and D.K. Renuka, "Improving the performance of aspect based sentiment analysis using fine-tuned Bert Base Uncased model," Int. Journal of Intelligent Networks, vol. 2, pp. 64-69, 2021.

In [35]:
# Load the model
model = AutoModelForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3
)

def compute_metrics(eval_pred):
    """
    Metrics function with F1-score to account for imbalanced classes.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    # Macro F1 treats all classes equally regardless of their frequency
    f1_macro = f1_score(labels, predictions, average='macro')

    # Weighted F1 accounts for class imbalance by weighting by support
    f1_weighted = f1_score(labels, predictions, average='weighted')

    return {
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

# Model configuration based on Geetha and Renuka (2021)
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/bert_topic_model',
    num_train_epochs=5,              # From literature
    per_device_train_batch_size=16,
    learning_rate=2e-5,              # From literature
    evaluation_strategy="epoch",     # Evaluate after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",  # Use weighted F1 for imbalanced data
    report_to="none"                 # Disable wandb and all other reporting!
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Fine-tune the pre-trained model

In [36]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

# Train the model
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted
1,0.086900,0.083768,0.913619,0.974779
2,0.057400,0.088201,0.934203,0.979286
3,0.025400,0.123071,0.933249,0.978915
4,0.008300,0.129264,0.941068,0.981088
5,0.005600,0.141893,0.935372,0.979296


TrainOutput(global_step=10065, training_loss=0.04416424992313387, metrics={'train_runtime': 3633.5168, 'train_samples_per_second': 44.31, 'train_steps_per_second': 2.77, 'total_flos': 1.0590315063552e+16, 'train_loss': 0.04416424992313387, 'epoch': 5.0})

## Make predictions for the test set and evaluate

We make predictions with the fine-tuned model (predict the labels of the documents in the test set) and evaluate the model's performance on the test set.

In [37]:
# Make predictions for test set
print("Evaluating on test set...")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate error rate
error_rate = 1 - (preds == labels).mean()
print(f"Error Rate: {error_rate:.4f}")

# Convert numeric predictions back to text labels for report
label_map_reverse = {v: k for k, v in topic_mapping.items()}
pred_labels = [label_map_reverse[p] for p in preds]
true_labels = [label_map_reverse[l] for l in labels]

# Display classification report
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels))

Evaluating on test set...


Error Rate: 0.4444

Classification Report:
              precision    recall  f1-score   support

        book       0.60      1.00      0.75         6
       movie       0.50      0.67      0.57         6
      sports       0.00      0.00      0.00         6

    accuracy                           0.56        18
   macro avg       0.37      0.56      0.44        18
weighted avg       0.37      0.56      0.44        18



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [38]:
# Save the model
print("Saving model...")
trainer.save_model('/content/drive/MyDrive/bert_topic_model/final')

Saving model...
